# Model Evaluation and Feature Engineering
## 1. Introduction

In Week 7, I learned the basic machine learning workflow:

**Split -> Train -> Predict -> Evaluate**

I implemented:

- Linear Regression for a regression problem.
- Logistic Regression for a binary classification problem.

In Week 8, I will improve the model evaluation process by using cross-validation and hyperparameter tuning.

I will also practice important feature-engineering techniques such as categorical encoding, feature scaling, and outlier handling.

The goal is not simply to obtain a higher score, but to evaluate models more carefully and understand how preprocessing, cross-validation, and hyperparameter tuning affect model performance.

### Importing libraries

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes, load_breast_cancer

from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.linear_model import LinearRegression, LogisticRegression

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully!")



Libraries imported successfully!


## 2. Regression Dataset

For consistency with Week 7, I will use the scikit-learn Diabetes dataset. The target variable represents a quantitative measure of disease progression. This allows me to compare the original Week 7 Linear Regression model with the cross-validated and tuned approach used in Week 8.

In [27]:
diabetes = load_diabetes()

X_reg = pd.DataFrame(
    diabetes.data,
    columns=diabetes.feature_names
)

y_reg = pd.Series(
    diabetes.target,
    name="target"
)

print("Regression features shape:", X_reg.shape)
print("Regression target shape:", y_reg.shape)
print("\nFeature names:")
print(list(X_reg.columns))

X_reg.head()

Regression features shape: (442, 10)
Regression target shape: (442,)

Feature names:
['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


## 3. Classification Dataset

For consistency with Week 7, I will use the scikit-learn Breast Cancer dataset. The target is binary, making it appropriate for Logistic Regression. This dataset will be used to compare the original Week 7 Logistic Regression model with the cross-validated and tuned model.

In [3]:
cancer = load_breast_cancer()

X_clf = pd.DataFrame(
    cancer.data,
    columns=cancer.feature_names
)

y_clf = pd.Series(
    cancer.target,
    name="target"
)

print("Classification features shape:", X_clf.shape)
print("Classification target shape:", y_clf.shape)

print("\nClass distribution:")
print(y_clf.value_counts())

X_clf.head()

Classification features shape: (569, 30)
Classification target shape: (569,)

Class distribution:
target
1    357
0    212
Name: count, dtype: int64


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 4. Baseline Regression Model

The baseline model represents the Week 7 approach.
I will use:
- Train/test split
- Linear Regression
- MAE
- MSE
- RMSE
- R²
This baseline will later be compared with the cross-validated and tuned model.

In [4]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

baseline_reg_model = LinearRegression()

baseline_reg_model.fit(X_train_reg, y_train_reg)

baseline_reg_predictions = baseline_reg_model.predict(X_test_reg)

baseline_reg_mae = mean_absolute_error(
    y_test_reg,
    baseline_reg_predictions
)

baseline_reg_mse = mean_squared_error(
    y_test_reg,
    baseline_reg_predictions
)

baseline_reg_rmse = np.sqrt(baseline_reg_mse)

baseline_reg_r2 = r2_score(
    y_test_reg,
    baseline_reg_predictions
)

print("Baseline Regression Results")
print("---------------------------")
print("MAE :", baseline_reg_mae)
print("MSE :", baseline_reg_mse)
print("RMSE:", baseline_reg_rmse)
print("R²  :", baseline_reg_r2)

Baseline Regression Results
---------------------------
MAE : 42.794094679599944
MSE : 2900.193628493482
RMSE: 53.85344583676593
R²  : 0.4526027629719195


## 5. Baseline Classification Model

The baseline classification model represents the Week 7 Logistic Regression approach.

The evaluation metrics are:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

These metrics provide more information than accuracy alone.

In [5]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

baseline_clf_model = LogisticRegression(
    max_iter=10000
)

baseline_clf_model.fit(X_train_clf, y_train_clf)

baseline_clf_predictions = baseline_clf_model.predict(X_test_clf)

baseline_accuracy = accuracy_score(
    y_test_clf,
    baseline_clf_predictions
)

baseline_precision = precision_score(
    y_test_clf,
    baseline_clf_predictions
)

baseline_recall = recall_score(
    y_test_clf,
    baseline_clf_predictions
)

baseline_f1 = f1_score(
    y_test_clf,
    baseline_clf_predictions
)

baseline_cm = confusion_matrix(
    y_test_clf,
    baseline_clf_predictions
)

print("Baseline Classification Results")
print("--------------------------------")
print("Accuracy :", baseline_accuracy)
print("Precision:", baseline_precision)
print("Recall   :", baseline_recall)
print("F1-score :", baseline_f1)

print("\nConfusion Matrix:")
print(baseline_cm)

Baseline Classification Results
--------------------------------
Accuracy : 0.9649122807017544
Precision: 0.9594594594594594
Recall   : 0.9861111111111112
F1-score : 0.9726027397260274

Confusion Matrix:
[[39  3]
 [ 1 71]]


## 6. Why Accuracy Alone Can Be Misleading

Accuracy measures the proportion of all predictions that are correct.

However, accuracy can hide poor performance on an important minority class.

For example, if 95% of observations belong to one class, a model that always predicts that majority class could achieve approximately 95% accuracy while completely failing to identify the minority class.

For this reason, classification should also be evaluated using precision, recall, F1-score, and the confusion matrix.

In this project, I will use multiple metrics rather than relying only on accuracy.

In [6]:
print("Baseline classification metrics:")
print(f"Accuracy : {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall   : {baseline_recall:.4f}")
print(f"F1-score : {baseline_f1:.4f}")

Baseline classification metrics:
Accuracy : 0.9649
Precision: 0.9595
Recall   : 0.9861
F1-score : 0.9726


## 7. K-Fold Cross-Validation

K-Fold Cross-Validation divides the training data into K approximately equal parts called folds.

The model is trained on K-1 folds and validated on the remaining fold.

This process is repeated until every fold has been used as the validation set.

The resulting scores can then be averaged.

This provides a more robust estimate of model performance than relying on only one train/test split.

In [7]:
kf_reg = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

reg_cv_scores = cross_val_score(
    LinearRegression(),
    X_train_reg,
    y_train_reg,
    cv=kf_reg,
    scoring="r2"
)

print("Regression Cross-Validation R² Scores:")
print(reg_cv_scores)

print("\nMean CV R²:", reg_cv_scores.mean())
print("Standard Deviation:", reg_cv_scores.std())

Regression Cross-Validation R² Scores:
[0.47012548 0.53681453 0.41108328 0.49128859 0.49251084]

Mean CV R²: 0.4803645434411371
Standard Deviation: 0.040885694891790315


### Classification of Cross-Validaion

In [10]:
skf_clf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

clf_cv_scores = cross_val_score(
    LogisticRegression(max_iter=10000),
    X_train_clf,
    y_train_clf,
    cv=skf_clf,
    scoring="f1"
)

print("Classification Cross-Validation F1 Scores:")
print(clf_cv_scores)

print("\nMean CV F1:", clf_cv_scores.mean())
print("Standard Deviation:", clf_cv_scores.std())

Classification Cross-Validation F1 Scores:
[0.97435897 0.93913043 0.95575221 0.96610169 0.94736842]

Mean CV F1: 0.9565423474997699
Standard Deviation: 0.012623575246896309


## 8. Feature Engineering

Feature engineering involves transforming raw variables into forms that are more suitable for machine learning.
In this section I will demonstrate three important preprocessing techniques:
1. Categorical encoding
2. Feature scaling
3. Outlier handling
   
The Week 7 datasets contain numerical features, so the categorical encoding example below uses a small illustrative dataset rather than artificially changing the original datasets.

In [11]:
categorical_data = pd.DataFrame({
    "gender": ["Female", "Male", "Female", "Male", "Female"],
    "city": ["Janakpur", "Kathmandu", "Birgunj", "Janakpur", "Kathmandu"],
    "age": [22, 35, 28, 41, 30]
})

categorical_data

,gender,city,age
0,Female,Janakpur,22
1,Male,Kathmandu,35
2,Female,Birgunj,28
3,Male,Janakpur,41
4,Female,Kathmandu,30


In [12]:
encoded_data = pd.get_dummies(
    categorical_data,
    columns=["gender", "city"],
    dtype=int
)

encoded_data

,age,gender_Female,gender_Male,city_Birgunj,city_Janakpur,city_Kathmandu
0,22,1,0,0,1,0
1,35,0,1,0,0,1
2,28,1,0,1,0,0
3,41,0,1,0,1,0
4,30,1,0,0,0,1


### Feature Scaling

Feature scaling places numerical features on comparable scales.

Standardization transforms a feature approximately so that it has:

- Mean = 0
- Standard deviation = 1

This is particularly useful for algorithms such as Logistic Regression that can be sensitive to feature scale.

In [13]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(
    X_train_clf
)

scaled_features_df = pd.DataFrame(
    scaled_features,
    columns=X_train_clf.columns
)

scaled_features_df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,-1.072001,-0.658425,-1.088080,-0.939274,-0.135940,-1.008718,-0.968359,-1.102032,0.281062,-0.113231,...,-1.034094,-0.623497,-1.070773,-0.876534,-0.169982,-1.038836,-1.078995,-1.350527,-0.352658,-0.541380
1,1.748743,0.066502,1.751157,1.745559,1.274468,0.842288,1.519852,1.994664,-0.293045,-0.320180,...,1.228342,-0.092833,1.187467,1.104386,1.517001,0.249655,1.178594,1.549916,0.191078,-0.173739
2,-0.974734,-0.931124,-0.997709,-0.867589,-0.613515,-1.138154,-1.092292,-1.243358,0.434395,-0.429247,...,-0.973231,-1.036772,-1.008044,-0.834168,-1.097823,-1.167260,-1.282241,-1.707442,-0.307734,-1.213033
3,-0.145103,-1.215186,-0.123013,-0.253192,0.664482,0.286762,-0.129729,-0.098605,0.555635,0.029395,...,-0.251266,-1.369643,-0.166633,-0.330292,0.234006,0.096874,-0.087521,-0.344838,0.242198,-0.118266
4,-0.771617,-0.081211,-0.803700,-0.732927,-0.672282,-1.006099,-0.798502,-0.684484,0.737495,-0.457213,...,-0.801135,0.079230,-0.824381,-0.741830,-0.911367,-0.984612,-0.933190,-0.777604,0.555118,-0.761639


### Outlier Handling

An outlier is an observation that is unusually far from the rest of the data.

One common way to identify outliers is the Interquartile Range (IQR) method.

IQR = Q3 - Q1

Potential outliers can be identified using:

Lower Bound = Q1 - 1.5 × IQR

Upper Bound = Q3 + 1.5 × IQR

Outlier handling should be performed carefully because extreme values may sometimes represent legitimate observations.

In [14]:
def find_outliers_iqr(dataframe, column):
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)
    
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = dataframe[
        (dataframe[column] < lower_bound) |
        (dataframe[column] > upper_bound)
    ]
    
    return outliers, lower_bound, upper_bound


outliers, lower_bound, upper_bound = find_outliers_iqr(
    X_reg,
    "bmi"
)

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of potential outliers:", len(outliers))

Lower bound: -0.13244469328909578
Upper bound: 0.1294636406639386
Number of potential outliers: 3


## 9. Hyperparameter Tuning with GridSearchCV

Hyperparameters are settings chosen before model training.

Instead of manually selecting values, GridSearchCV evaluates different combinations using cross-validation.

For Logistic Regression, the `C` parameter controls the strength of regularization.

I will test several possible values of `C` and use 5-fold stratified cross-validation to identify the best-performing combination based on F1-score.

In [16]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=10000))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid_search_clf = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search_clf.fit(
    X_train_clf,
    y_train_clf
)

print("Best parameters:")
print(grid_search_clf.best_params_)

print("\nBest cross-validation F1-score:")
print(grid_search_clf.best_score_)

Best parameters:
{'model__C': 0.1}

Best cross-validation F1-score:
0.984524686809138


### Evaluating the Tuned Classification Model

In [17]:
tuned_clf_model = grid_search_clf.best_estimator_

tuned_clf_predictions = tuned_clf_model.predict(
    X_test_clf
)

tuned_accuracy = accuracy_score(
    y_test_clf,
    tuned_clf_predictions
)

tuned_precision = precision_score(
    y_test_clf,
    tuned_clf_predictions
)

tuned_recall = recall_score(
    y_test_clf,
    tuned_clf_predictions
)

tuned_f1 = f1_score(
    y_test_clf,
    tuned_clf_predictions
)

tuned_cm = confusion_matrix(
    y_test_clf,
    tuned_clf_predictions
)

print("Tuned Classification Results")
print("-----------------------------")
print("Accuracy :", tuned_accuracy)
print("Precision:", tuned_precision)
print("Recall   :", tuned_recall)
print("F1-score :", tuned_f1)

print("\nConfusion Matrix:")
print(tuned_cm)

Tuned Classification Results
-----------------------------
Accuracy : 0.9736842105263158
Precision: 0.9726027397260274
Recall   : 0.9861111111111112
F1-score : 0.9793103448275862

Confusion Matrix:
[[40  2]
 [ 1 71]]


### Tuning the Regression Model


In [18]:
from sklearn.linear_model import Ridge

## 10. Tuned Regression Model

The Week 7 baseline used ordinary Linear Regression.

For the improved regression model, I will use Ridge Regression, a regularized form of linear regression.

The `alpha` hyperparameter controls the strength of regularization.

GridSearchCV will test several values using 5-fold cross-validation.

In [19]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

ridge_param_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100]
}

grid_search_reg = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid_search_reg.fit(
    X_train_reg,
    y_train_reg
)

print("Best parameters:")
print(grid_search_reg.best_params_)

print("\nBest cross-validation R²:")
print(grid_search_reg.best_score_)

Best parameters:
{'model__alpha': 10}

Best cross-validation R²:
0.453937415795935


### Evaluating the Tuned Regression

In [20]:
tuned_reg_model = grid_search_reg.best_estimator_

tuned_reg_predictions = tuned_reg_model.predict(
    X_test_reg
)

tuned_reg_mae = mean_absolute_error(
    y_test_reg,
    tuned_reg_predictions
)

tuned_reg_mse = mean_squared_error(
    y_test_reg,
    tuned_reg_predictions
)

tuned_reg_rmse = np.sqrt(
    tuned_reg_mse
)

tuned_reg_r2 = r2_score(
    y_test_reg,
    tuned_reg_predictions
)

print("Tuned Regression Results")
print("------------------------")
print("MAE :", tuned_reg_mae)
print("MSE :", tuned_reg_mse)
print("RMSE:", tuned_reg_rmse)
print("R²  :", tuned_reg_r2)

Tuned Regression Results
------------------------
MAE : 42.856825247800614
MSE : 2875.778718421843
RMSE: 53.626287568895194
R²  : 0.45721095677808476


### Outlier Handling

Outliers are observations that are unusually far from the rest of the data.

One common method for detecting outliers is the Interquartile Range (IQR) method.

Instead of deleting observations, I will demonstrate outlier capping.

In outlier capping, values below the lower IQR boundary are replaced by the lower boundary, while values above the upper boundary are replaced by the upper boundary.

This preserves the number of observations while reducing the influence of extreme values.

Outlier handling should always be performed carefully because an extreme value may represent a legitimate observation.

In [23]:
X_reg_capped = X_reg.copy()

q1 = X_reg_capped["bmi"].quantile(0.25)
q3 = X_reg_capped["bmi"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

original_min = X_reg_capped["bmi"].min()
original_max = X_reg_capped["bmi"].max()

X_reg_capped["bmi"] = X_reg_capped["bmi"].clip(
    lower=lower_bound,
    upper=upper_bound
)

capped_min = X_reg_capped["bmi"].min()
capped_max = X_reg_capped["bmi"].max()

print("Original BMI minimum:", original_min)
print("Original BMI maximum:", original_max)

print("\nIQR lower bound:", lower_bound)
print("IQR upper bound:", upper_bound)

print("\nAfter outlier capping:")
print("New BMI minimum:", capped_min)
print("New BMI maximum:", capped_max)

Original BMI minimum: -0.09027529589850945
Original BMI maximum: 0.17055522598064407

IQR lower bound: -0.13244469328909578
IQR upper bound: 0.1294636406639386

After outlier capping:
New BMI minimum: -0.09027529589850945
New BMI maximum: 0.1294636406639386


## 11. Before vs After Model Comparison

The baseline models represent the Week 7 approach.

The tuned models use preprocessing, cross-validation, and hyperparameter tuning.

The comparison below shows how the evaluation results changed.

In [21]:
# Regression Comparison
regression_comparison = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²"],
    "Week 7 Baseline": [
        baseline_reg_mae,
        baseline_reg_mse,
        baseline_reg_rmse,
        baseline_reg_r2
    ],
    "Week 8 Tuned": [
        tuned_reg_mae,
        tuned_reg_mse,
        tuned_reg_rmse,
        tuned_reg_r2
    ]
})

regression_comparison

,Metric,Week 7 Baseline,Week 8 Tuned
0,MAE,42.794095,42.856825
1,MSE,2900.193628,2875.778718
2,RMSE,53.853446,53.626288
3,R²,0.452603,0.457211


In [22]:
# Classification Comparison
classification_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Week 7 Baseline": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1
    ],
    "Week 8 Tuned": [
        tuned_accuracy,
        tuned_precision,
        tuned_recall,
        tuned_f1
    ]
})

classification_comparison

,Metric,Week 7 Baseline,Week 8 Tuned
0,Accuracy,0.964912,0.973684
1,Precision,0.959459,0.972603
2,Recall,0.986111,0.986111
3,F1-score,0.972603,0.979310


## 12. Classification Confusion Matrix Comparison

The confusion matrix shows how many observations were correctly and incorrectly classified.

Comparing the baseline and tuned confusion matrices helps determine whether changes in the model affected false positives and false negatives.

In [24]:
print("Week 7 Baseline Confusion Matrix")
print("--------------------------------")
print(baseline_cm)

print("\nWeek 8 Tuned Confusion Matrix")
print("-----------------------------")
print(tuned_cm)

Week 7 Baseline Confusion Matrix
--------------------------------
[[39  3]
 [ 1 71]]

Week 8 Tuned Confusion Matrix
-----------------------------
[[40  2]
 [ 1 71]]


## 12. Interpretation of Results

The Week 7 models provide the baseline performance using a single train/test split.

In Week 8, cross-validation provides a more robust estimate of model performance by evaluating the model across multiple folds.

Feature scaling was incorporated into the classification pipeline because Logistic Regression can be affected by differences in feature scale.

GridSearchCV tested multiple hyperparameter values using cross-validation and selected the best-performing configuration according to the chosen evaluation metric.

The tuned model results are compared with the Week 7 baseline results above.

If the tuned model has better evaluation metrics on the held-out test set, this suggests that the preprocessing and hyperparameter tuning helped performance for this particular split.

If some metrics are similar or lower, that is also an important result. Hyperparameter tuning does not guarantee improvement on every test set. Its main purpose is to select model settings systematically using cross-validation rather than manual guessing.

In [25]:
print("Regression Metric Changes")
print("-------------------------")

print("MAE change :", tuned_reg_mae - baseline_reg_mae)
print("MSE change :", tuned_reg_mse - baseline_reg_mse)
print("RMSE change:", tuned_reg_rmse - baseline_reg_rmse)
print("R² change  :", tuned_reg_r2 - baseline_reg_r2)

print("\nClassification Metric Changes")
print("-----------------------------")

print("Accuracy change :", tuned_accuracy - baseline_accuracy)
print("Precision change:", tuned_precision - baseline_precision)
print("Recall change   :", tuned_recall - baseline_recall)
print("F1-score change :", tuned_f1 - baseline_f1)

Regression Metric Changes
-------------------------
MAE change : 0.06273056820067069
MSE change : -24.414910071638587
RMSE change: -0.22715826787073468
R² change  : 0.004608193806165284

Classification Metric Changes
-----------------------------
Accuracy change : 0.00877192982456143
Precision change: 0.01314328026656797
Recall change   : 0.0
F1-score change : 0.006707605101558767


## 13. Short Summary

In Week 8, I learned that evaluating a machine learning model using only one train/test split can provide a limited view of its performance.

I practiced K-Fold Cross-Validation to evaluate models across multiple folds.

I also practiced feature engineering techniques including categorical encoding, feature scaling, and identifying potential outliers using the IQR method.

For hyperparameter tuning, I used GridSearchCV to systematically test different parameter values using cross-validation.

I re-ran the Week 7 regression and classification projects and compared the baseline models with tuned models.

The comparison shows how cross-validation, preprocessing, and hyperparameter tuning can affect model performance. The results also demonstrate that improvement should be evaluated using appropriate metrics rather than assumed in advance.

# Week 8 Completion Checklist

-  Explained why raw accuracy can be misleading.
-  Explained K-Fold Cross-Validation.
-  Applied K-Fold Cross-Validation to regression.
-  Applied Stratified Cross-Validation to classification.
-  Demonstrated categorical encoding.
-  Demonstrated feature scaling.
-  Identified potential outliers using the IQR method.
-  Demonstrated outlier handling using capping.
-  Used GridSearchCV for hyperparameter tuning.
-  Re-ran the Week 7 regression project.
-  Re-ran the Week 7 classification project.
-  Compared baseline and tuned regression results.
-  Compared baseline and tuned classification results.
-  Compared confusion matrices.
-  Included interpretation of the results.
-  Included a short final summary.

This notebook contains the Week 8 model evaluation and feature-engineering work, including baseline models, cross-validation, preprocessing, outlier handling, hyperparameter tuning, before-and-after comparisons, and written interpretation.